# Superstore Orders: Data Cleaning and Feature Engineering
Dataset: Kaggle Superstore orders (51,290 rows, 21 columns, 2011 to 2014).

**Contents**
1. Setup and load
2. Data cleaning
3. Data validation
4. Feature engineering
5. Final check and save

## 1. Setup and load
All columns are loaded as text first, so each one can be converted deliberately.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("SuperStoreOrders.csv", encoding="utf-8-sig", dtype=str)

## 2. Data cleaning

### 2.1 Sales: text to integer
Sales is stored as text because 2,630 values contain commas (for example `1,648`). The commas are removed before converting.

In [3]:
# Clean sales: remove commas, then convert
df["sales"] = pd.to_numeric(
    df["sales"].str.strip().str.replace(",", "", regex=False),
    errors="coerce"
).astype("float64")

print("Missing sales:", df["sales"].isna().sum())

Missing sales: 0


In [4]:
df["sales"] = df["sales"].astype("int64")
print(df["sales"].dtype)

int64


### 2.2 Other numeric columns

In [5]:
num_cols = ["quantity", "discount", "profit", "shipping_cost"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col].str.strip(), errors="coerce")
    print(col, "missing:", df[col].isna().sum())

quantity missing: 0
discount missing: 0
profit missing: 0
shipping_cost missing: 0


### 2.3 Dates
Dates come in two patterns (`13-01-2011` and `1/1/2011`), both day first. Dashes are replaced with slashes so one format covers all rows.

In [6]:
for col in ["order_date","ship_date"]:
    df[col] = df[col].str.replace("-", "/", regex = False)

In [7]:
for col in ["order_date", "ship_date"]:
    df[col] = pd.to_datetime(df[col], format = "%d/%m/%Y", errors = "coerce")

### 2.4 Order IDs

In [8]:
df["order_id"] = df["order_id"].str.strip().str.upper()
print("Blank order IDs:", (df["order_id"] == "").sum() + df["order_id"].isna().sum())

Blank order IDs: 0


### 2.5 Year column
The `year` column was stored as text, so it is converted to `int64` before it is compared with the year of `order_date`.

In [9]:
df["year"] = df["year"].astype("int64")

## 3. Data validation

### 3.1 Date and value checks

In [10]:
print("Ship before order:", (df["ship_date"] < df["order_date"]).sum())

Ship before order: 0


In [11]:
print("Negative sales:", (df["sales"] < 0).sum())
print("Negative quantity:", (df["quantity"] < 0).sum())
print("Discount outside 0-1:", ((df["discount"] < 0) | (df["discount"] > 1)).sum())

Negative sales: 0
Negative quantity: 0
Discount outside 0-1: 0


### 3.2 Missing values and summary statistics

In [12]:
print(df[["sales", "quantity", "discount", "profit", "shipping_cost"]].describe())
print(df.isna().sum())

              sales      quantity      discount        profit  shipping_cost
count  51290.000000  51290.000000  51290.000000  51290.000000   51290.000000
mean     246.498440      3.476545      0.142908     28.641740      26.375915
std      487.567175      2.278766      0.212280    174.424113      57.296804
min        0.000000      1.000000      0.000000  -6599.978000       0.000000
25%       31.000000      2.000000      0.000000      0.000000       2.610000
50%       85.000000      3.000000      0.000000      9.240000       7.790000
75%      251.000000      5.000000      0.200000     36.810000      24.450000
max    22638.000000     14.000000      0.850000   8399.976000     933.570000
order_id          0
order_date        0
ship_date         0
ship_mode         0
customer_name     0
segment           0
state             0
country           0
market            0
region            0
product_id        0
category          0
sub_category      0
product_name      0
sales             0
quantit

### 3.3 Duplicates
There are no exact duplicates. 38 Order ID and Product ID pairs repeat, but they differ in quantity or shipping cost, so they are separate line items and are kept.

In [13]:
print("Exact duplicates:", df.duplicated().sum())

Exact duplicates: 0


In [14]:
dupe_rows = df[df.duplicated(["order_id", "product_id"], keep=False)]

In [15]:
len(dupe_rows), dupe_rows["order_id"].nunique()

(76, 38)

### 3.4 Year cross-check
The converted order dates must agree with the dataset's `year` column.

In [16]:
(df["order_date"].dt.year == df["year"]).all()

np.True_

## 4. Feature engineering
Every new column is calculated from columns already in the file.

### 4.1 Delivery days

In [17]:
df["delivery_days"] = (df["ship_date"] - df["order_date"]).dt.days
print(df["delivery_days"].min(), df["delivery_days"].max())

0 7


### 4.2 Month and quarter

In [18]:
df["order_month"] = df["order_date"].dt.month
df["order_quarter"] = df["order_date"].dt.quarter

### 4.3 Profit margin
Left empty for the one row where Sales is 0, to avoid dividing by zero.

In [19]:
df["profit_margin"] = df["profit"] / df["sales"]
df.loc[df["sales"] == 0, "profit_margin"] = float("nan")

## 5. Final check and save

In [20]:
print(df.shape)
print(df["order_date"].isna().sum(), df["ship_date"].isna().sum())
print(df["sales"].dtype, df["sales"].sum())

(51290, 25)
0 0
int64 12642905


In [21]:
df.to_csv("superstore_clean.csv", index=False)